### Imports

In [69]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

In [70]:
df = pd.read_csv('data/clean/mp_shots_cleaner.csv')

### Downcast

In [71]:
int_shots = df.select_dtypes('int64').columns
float_shots = df.select_dtypes('float64').columns

df[int_shots] = df[int_shots].apply(pd.to_numeric, downcast='integer')
df[float_shots] = df[float_shots].apply(pd.to_numeric, downcast='float')

obj_shots = df.select_dtypes(include=['object']).columns

for col in obj_shots:
    num_unique = df[col].nunique()
    num_total = len(df[col])
    
    if num_unique / num_total < 0.5:
        df[col] = df[col].astype('category')

In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1028007 entries, 0 to 1028006
Data columns (total 88 columns):
 #   Column                                                 Non-Null Count    Dtype   
---  ------                                                 --------------    -----   
 0   xcordadjusted                                          1028007 non-null  int8    
 1   shotangle                                              1028007 non-null  float32 
 2   defendingteammaxtimeonicesincefaceoff                  1028007 non-null  float32 
 3   playerpositionthatdidevent                             1028007 non-null  category
 4   shootingteammintimeoniceofforwardssincefaceoff         1028007 non-null  float32 
 5   defendingteamaveragetimeoniceofdefencemensincefaceoff  1028007 non-null  float32 
 6   shotanglereboundroyalroad                              1028007 non-null  int8    
 7   time                                                   1028007 non-null  int16   
 8   shootingteam

In [73]:
df.nunique().sort_values().head(40)

shotonemptynet                                        2
goal                                                  2
shotwasongoal                                         2
homeemptynet                                          2
offwing                                               2
shotrebound                                           2
shotrush                                              2
ishometeam                                            2
awayemptynet                                          2
shotanglereboundroyalroad                             2
location                                              3
shootingteamdefencemenonice                           4
period                                                4
defendingteamdefencemenonice                          4
playerpositionthatdidevent                            5
awaypenalty1length                                    5
homepenalty1length                                    6
defendingteamforwardsonice                      

In [74]:
# Fix lasteventteam values
df = df[df['lasteventteam'].isin(['HOME', 'AWAY'])]
df['lasteventhometeam'] = df['lasteventteam'] == 'HOME'
df.drop(columns=['lasteventteam'])

,xcordadjusted,shotangle,defendingteammaxtimeonicesincefaceoff,playerpositionthatdidevent,shootingteammintimeoniceofforwardssincefaceoff,defendingteamaveragetimeoniceofdefencemensincefaceoff,shotanglereboundroyalroad,time,shootingteamdefencemenonice,homepenalty1length,...,lasteventycord,defendingteammintimeoniceofforwards,shotdistance,xcord,location,homeskatersonice,shootertimeonicesincefaceoff,shotangleplusreboundspeed,shootingteammaxtimeoniceofdefencemensincefaceoff,lasteventhometeam
0,59,-40.914383,26.0,D,20.0,26.0,0,61,2,0,...,30,26.0,39.698868,59,AWAYZONE,5,21,0.000000,21.0,False
1,81,45.000000,35.0,C,29.0,35.0,1,70,2,0,...,-26,35.0,11.313708,81,AWAYZONE,5,29,9.546042,30.0,True
2,55,41.423664,26.0,L,19.0,26.0,0,107,2,0,...,-40,19.0,45.343136,55,AWAYZONE,5,19,0.000000,19.0,True
3,58,-44.060810,47.0,C,28.0,45.0,0,177,2,0,...,-27,26.0,43.139309,58,AWAYZONE,5,28,0.000000,39.0,False
4,64,-53.673176,48.0,L,46.0,40.5,0,231,3,0,...,-38,46.0,42.201897,-64,HOMEZONE,5,46,0.000000,51.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1028002,6,15.488502,10.0,C,10.0,10.0,0,3529,2,0,...,-22,24.0,86.127815,-6,Neu. Zone,6,10,0.000000,10.0,True
1028003,74,-56.888657,65.0,C,8.0,65.0,0,3590,2,0,...,-29,4.0,27.459061,-74,HOMEZONE,6,8,0.000000,65.0,True
1028004,3,-17.429863,20.0,R,20.0,20.0,0,3508,2,0,...,-22,20.0,90.138779,3,Neu. Zone,6,20,0.000000,20.0,False
1028005,6,19.259291,9.0,R,9.0,9.0,0,3443,2,0,...,-19,9.0,87.920418,-6,Neu. Zone,6,9,6.419764,9.0,True


In [75]:
df['lasteventhometeam']

0          False
1           True
2           True
3          False
4          False
           ...  
1028002     True
1028003     True
1028004    False
1028005     True
1028006    False
Name: lasteventhometeam, Length: 1027818, dtype: bool

### Get Dummies

In [76]:
cat_list = [col for col in df.columns if 3 <= df[col].nunique() <= 20]
cat_df = df[cat_list]
for col in cat_df.columns:
    counts = df[col].value_counts()
    rare = counts[counts < 100].index
    df[col] = df[col].replace(rare, "OTHER")

/var/folders/kr/vdj9mkvn5wv12wpg83j9kl1w0000gn/T/ipykernel_36384/273117029.py:6: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df[col] = df[col].replace(rare, "OTHER")
/var/folders/kr/vdj9mkvn5wv12wpg83j9kl1w0000gn/T/ipykernel_36384/273117029.py:6: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df[col] = df[col].replace(rare, "OTHER")


In [77]:
cat_df.head()

,playerpositionthatdidevent,shootingteamdefencemenonice,homepenalty1length,defendingteamforwardsonice,shottype,awayteamgoals,hometeamgoals,lasteventcategory,period,defendingteamdefencemenonice,awayskatersonice,awaypenalty1length,shootingteamforwardsonice,location,homeskatersonice
0,D,2,0,3,WRIST,0,0,GIVE,1,2,5,0,3,AWAYZONE,5
1,C,2,0,3,TIP,0,0,SHOT,1,2,5,0,3,AWAYZONE,5
2,L,2,0,3,SNAP,0,0,HIT,1,2,5,0,3,AWAYZONE,5
3,C,2,0,3,WRIST,0,0,TAKE,1,2,5,0,3,AWAYZONE,5
4,L,3,0,3,WRIST,0,0,HIT,1,2,5,0,2,HOMEZONE,5


In [78]:
# Drop non categorical variables
noncat = ['shootingteamdefencemenonice', 'defendingteamforwardsonice', 'awayteamgoals', 'hometeamgoals',
          'defendingteamdefencemenonice', 'awayskatersonice', 'shootingteamforwardsonice', 'homeskatersonice']
new_cat_list = [x for x in cat_list if x not in noncat]
new_cat_list

['playerpositionthatdidevent',
 'homepenalty1length',
 'shottype',
 'lasteventcategory',
 'period',
 'awaypenalty1length',
 'location']

In [79]:
df = pd.get_dummies(df, columns=new_cat_list)

In [80]:
df.head()

,xcordadjusted,shotangle,defendingteammaxtimeonicesincefaceoff,shootingteammintimeoniceofforwardssincefaceoff,defendingteamaveragetimeoniceofdefencemensincefaceoff,shotanglereboundroyalroad,time,shootingteamdefencemenonice,lasteventshotdistance,shootingteamaveragetimeoniceofdefencemen,...,period_3,period_4,awaypenalty1length_0,awaypenalty1length_120,awaypenalty1length_240,awaypenalty1length_300,awaypenalty1length_600,location_AWAYZONE,location_HOMEZONE,location_Neu. Zone
0,59,-40.914383,26.0,20.0,26.0,0,61,2,0.000000,20.500000,...,False,False,True,False,False,False,False,True,False,False
1,81,45.000000,35.0,29.0,35.0,1,70,2,39.698868,29.500000,...,False,False,True,False,False,False,False,True,False,False
2,55,41.423664,26.0,19.0,26.0,0,107,2,0.000000,19.000000,...,False,False,True,False,False,False,False,True,False,False
3,58,-44.060810,47.0,28.0,45.0,0,177,2,0.000000,38.500000,...,False,False,True,False,False,False,False,True,False,False
4,64,-53.673176,48.0,46.0,40.5,0,231,3,0.000000,17.666666,...,False,False,True,False,False,False,False,False,True,False


### Scaling

In [81]:
num_cols = df.select_dtypes(include='number').columns

# Making a Scaler object
scaler = preprocessing.StandardScaler()
# Fitting data to the scaler object
scaled_df = scaler.fit_transform(df[num_cols])
scaled_df = pd.DataFrame(scaled_df, columns=num_cols, index=df.index)

In [82]:
scaled_df.head()

,xcordadjusted,shotangle,defendingteammaxtimeonicesincefaceoff,shootingteammintimeoniceofforwardssincefaceoff,defendingteamaveragetimeoniceofdefencemensincefaceoff,shotanglereboundroyalroad,time,lasteventshotdistance,shootingteamaveragetimeoniceofdefencemen,awaypenalty1timeleft,...,defendingteammintimeonice,ycordadjusted,shotrush,lasteventycord,defendingteammintimeoniceofforwards,shotdistance,xcord,shootertimeonicesincefaceoff,shotangleplusreboundspeed,shootingteammaxtimeoniceofdefencemensincefaceoff
0,-0.083937,-1.076084,-0.442366,-0.179262,-0.218087,-0.330134,-1.682445,-0.443275,-0.440762,-0.279698,...,0.184185,-1.349757,-0.042025,1.313135,0.013251,0.288041,0.933359,-0.339830,-0.269858,-0.325595
1,1.090435,1.212323,-0.038793,0.365256,0.252978,3.029077,-1.673847,2.062021,-0.031571,-0.279698,...,0.730037,0.411380,-0.042025,-1.130550,0.514919,-1.208695,1.280318,0.110761,0.819627,0.156905
2,-0.297460,1.117064,-0.442366,-0.239764,-0.218087,-0.330134,-1.638498,-0.443275,-0.508960,-0.279698,...,-0.240367,1.550939,-0.042025,-1.741471,-0.376934,0.585661,0.870275,-0.452478,-0.269858,-0.432818
3,-0.137318,-1.159892,0.499305,0.304754,0.776384,-0.330134,-1.571621,-0.443275,0.377619,-0.279698,...,0.184185,-1.556949,-0.042025,-1.174187,0.013251,0.469454,0.917588,0.054437,-0.269858,0.639406
4,0.182966,-1.415926,0.544146,1.393791,0.540851,-0.330134,-1.520030,-0.443275,-0.569581,-0.279698,...,0.851337,-1.764142,-0.042025,-1.654197,1.128067,0.420025,-1.006459,1.068268,-0.269858,1.282740


### Train Test Split

In [83]:
X = df.drop('goal', axis=1)
y= df['goal']

In [84]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=12)

In [86]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((770863, 121), (256955, 121), (770863,), (256955,))